In [ ]:
!huggingface-cli login



    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) Y
Token is valid (permission: read).
The token `verilog` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pu

In [ ]:
messages = [
    {"role": "system", "content": "Generate a verification testbench for this Verilog module. Only give me the testbench and nothing else"},
    {"role": "user", "content": "module mux (\n    input [31:0] in0, in1,\n    input sel,\n    output [31:0] out\n);\n\n  assign out = sel ? in1 : in0;\n\nendmodule"}
]

outputs = pipeline(
    messages,
    max_new_tokens=256,
)
print(outputs[0]["generated_text"][-1])

{'role': 'assistant', 'content': 'module tb;\n\n  // Inputs\n  reg [31:0] in0;\n  reg [31:0] in1;\n  reg sel;\n\n  // Output\n  wire [31:0] out_ref;\n  wire [31:0] out_dut;\n\n  // Reference module\n  module reference (\n    input [31:0] in0, in1,\n    input sel,\n    output [31:0] out\n  );\n    assign out = sel? in1 : in0;\n  endmodule\n\n  // Instantiate the module\n  initial begin\n    $dumpfile("wave.vcd");\n    $dumpvars(1, tb );\n  end\n\n  initial begin\n    #5 $finish;\n  end\n\n  // Clock\n  initial forever\n    sel <= $random;\n\n  // Set up verification\n  reg [31:0] out_ref_next;\n  initial out_ref_next = 0;\n\n  task wait_for_stable(input[31:0] out_ref);\n    repeat(100) begin\n      @(negedge $time);\n      if (out_ref!== out_ref_next) out_ref_next = out_ref;\n      else break;\n    end\n  endtask\n\n\n  initial begin\n    repeat'}


In [ ]:
from huggingface_hub import InferenceClient

client = InferenceClient(base_url="http://localhost:8080/v1/")

response = client.chat.completions.create(
    model="tgi",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is deep learning?"}
    ],
    stream=True,
    max_tokens=20,
)

for chunk in response:
    print(chunk.choices[0].delta.content, end="")


In [ ]:
def inference(module_code):
  messages = [
      {"role": "system", "content": "Generate a verification testbench for this Verilog module. Only give me the testbench and nothing else"},
      {"role": "user", "content": module_code}
  ]

  outputs = pipeline(
      messages,
      max_new_tokens=650,
  )
  return outputs[0]["generated_text"][-1]["content"]

In [ ]:
import json


# Path to the input JSONL file
input_file_path = 'test_data.jsonl'

# Path to the output JSONL file
output_file_path = 'output_file.jsonl'


# Reading the input JSONL file and writing to the output JSONL file
with open(input_file_path, 'r') as infile, open(output_file_path, 'w') as outfile:
    line_count = 0
    for line in infile:
        line_count += 1
        print(f"Processing line {line_count}...")

        record = json.loads(line)  # Parse each line
        attribute1 = record['testbench']  # Extract one attribute
        attribute2 = inference(record['module_code'])  # Process another attribute

        # Prepare the new record with processed data
        new_record = {
            'gold_testbench': attribute1,
            'testbench': attribute2
        }

        # Write the new record as a JSON object to the output file
        outfile.write(json.dumps(new_record) + '\n')

print(f"Processed data has been written to {output_file_path}.")


Processing line 1...
Processing line 2...
Processing line 3...


KeyboardInterrupt: 

In [ ]:
import json
from google.colab import drive
import time

# Mount Google Drive
drive.mount('/content/drive')

# Path to the input JSONL file
input_file_path = 'test_data.jsonl'

# Path to save the output JSONL file in Google Drive
output_file_path = '/content/drive/My Drive/output_file.jsonl'

# Reset the output file (truncate contents)
open(output_file_path, 'w').close()

def process(value):
    # Modify this function as needed
    return value.upper()

# Reading the input JSONL file and writing to the output JSONL file
with open(input_file_path, 'r') as infile, open(output_file_path, 'a') as outfile:  # 'a' mode to append after resetting
    count = 0  # Counter to track iterations

    for line in infile:
        time.sleep(1.5)
        record = json.loads(line)  # Parse each line
        attribute1 = record['testbench']  # Extract one attribute
        attribute2 = process(record['module_code'])  # Process another attribute

        # Prepare the new record with processed data
        new_record = {
            'gold_testbench': attribute1,
            'testbench': attribute2
        }

        # Write the new record as a JSON object to the output file
        outfile.write(json.dumps(new_record) + '\n')
        count += 1

        # Save progress every 10 iterations
        if count % 10 == 0:
            outfile.flush()  # Save progress to disk
            print(f"Processed {count} records so far... Progress saved to Google Drive.")

# Ensure the final data is saved
outfile.flush()
print(f"Final output file saved to Google Drive at: {output_file_path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Processed 10 records so far... Progress saved to Google Drive.
Processed 20 records so far... Progress saved to Google Drive.
Processed 30 records so far... Progress saved to Google Drive.


ValueError: I/O operation on closed file.